# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name')}: {getattr(metadata, 'description')}")


## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields from the Croissant schema.

In [ ]:
# List all record sets and their fields/columns by @id
print("Available record sets and fields (by @id):\n")
record_sets = getattr(metadata, 'record_set', [])  # record_sets is a list (can be empty if none detected)
if not record_sets:
    print("No record sets were found in the Croissant schema.\nPlease check the schema details in the documentation or metadata.")
else:
    for record_set in record_sets:
        rs_id = getattr(record_set, '@id', None)
        rs_name = getattr(record_set, 'name', '')
        print(f"- RecordSet @id: {rs_id}, name: {rs_name}")
        if hasattr(record_set, 'field'):
            fields = getattr(record_set, 'field', [])
            for field in fields:
                print(f"    - Field @id: {getattr(field, '@id', None)}, name: {getattr(field, 'name', '')}, dataType: {getattr(field, 'data_type', '')}")
        elif hasattr(record_set, 'column'):
            columns = getattr(record_set, 'column', [])
            for column in columns:
                print(f"    - Column @id: {getattr(column, '@id', None)}, name: {getattr(column, 'name', '')}, dataType: {getattr(column, 'data_type', '')}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

**Note:** In this dataset, the Croissant metadata uses the `recordSet` term, but the actual record set(s) may reside under distribution(s) or as file objects. We'll try to auto-detect available record sets. If none, we'll demonstrate extracting the main dataset from available sources.

In [ ]:
# For this dataset, record sets may not be explicitly defined.
# We'll try to use the single file-based record set available, if any, or demonstrate with the main dataset file.
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

record_sets = getattr(metadata, 'record_set', None)

# Detect record set IDs. If none, use the primary distribution file for demonstration.
record_set_ids = []
if record_sets and len(record_sets):
    for record_set in record_sets:
        rs_id = getattr(record_set, '@id', None)
        if rs_id:
            record_set_ids.append(rs_id)

# If no record sets, try to infer from available distribution/datafile objects
if not record_set_ids:
    # Try to detect file-based record set
    if hasattr(metadata, 'distribution'):
        print("No explicit record set found. Available distributions:")
        for d in getattr(metadata, 'distribution', []):
            print(f"Distribution @id: {getattr(d, '@id', None)}, name: {getattr(d, 'name', '')}, encodingFormat: {getattr(d, 'encoding_format', '')}, contentUrl: {getattr(d, 'content_url', '')}")
    print("For this demo we attempt to autodetect record set from the first available file (if possible).")
    # For the notebook, we will try to call list(dataset.records())

# Try extracting records
dataframes = {}
try:
    # If record set ids found, extract by them
    if record_set_ids:
        for rs_id in record_set_ids:
            records = list(dataset.records(record_set=rs_id))
            dataframes[rs_id] = pd.DataFrame(records)
        # Use first record set as main
        main_rs_id = record_set_ids[0]
        print(f"Loaded DataFrame for record set {main_rs_id}.")
        print(f"Available columns: {dataframes[main_rs_id].columns.tolist()}")
        display(dataframes[main_rs_id].head())
    else:
        # No record sets: use default records
        all_records = list(dataset.records())  # This yields records from the primary data file
        if len(all_records):
            df = pd.DataFrame(all_records)
            main_rs_id = 'default_records'
            dataframes[main_rs_id] = df
            print(f"Loaded DataFrame from default dataset records.")
            print(f"Available columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print("No records found in the dataset. Please check the dataset file or record set definitions.")
except Exception as e:
    print(f"Could not extract records: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations in this section include removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**All field references use their `@id`.**

In [ ]:
# Choose a numeric field for analysis (by @id or fallback to column name if not found)
# List available columns
main_df_key = list(dataframes.keys())[0]  # Use primary loaded DataFrame
df = dataframes[main_df_key]
print(f"Available columns for EDA: {df.columns.tolist()}")

# Try to pick a numeric field likely in proteomics data, e.g. 'MW [kDa]', 'Abundance', 'Peptides', etc.
# By convention, in Croissant, the field @id would be e.g. 'MW[kDa]' or similar if not mapped.
candidate_numeric_fields = [col for col in df.columns if df[col].dtype in ('int64', 'float64') or pd.api.types.is_numeric_dtype(df[col])]
if not candidate_numeric_fields:
    # Try to find any column containing 'abundance', 'count', 'MW', 'quant', etc.
    for col in df.columns:
        for kw in ['abundance', 'count', 'MW', 'peptide', 'quant']:
            if kw.lower() in col.lower():
                candidate_numeric_fields.append(col)
                break
print(f"Candidate numeric fields: {candidate_numeric_fields}")

if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")
    
    # Remove NA, filter for values > threshold
    threshold = df[numeric_field].dropna().mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Try to find a group field (categorical), e.g. 'description', 'accession', 'Gene', 'Protein', etc.
    candidate_group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
    group_field = candidate_group_fields[0] if candidate_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field].mean().sort_values(numeric_field, ascending=False)
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if candidate_numeric_fields:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot grouped by group_field, if available and there are few unique categories
    if group_field and df[group_field].nunique() < 20:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric fields found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


- In this notebook, we demonstrated how to load and explore the FAIR²-compliant dataset on mass spectrometry analysis of extracellular vesicles using the `mlcroissant` library.
- The dataset was loaded via its Croissant schema, and key numeric and categorical attributes were explored by their `@id` or column name.
- We showed methods for filtering, normalizing, grouping, and visualizing proteomic data.

Further downstream tasks (e.g., machine learning or advanced normalization) can be performed by extending this workflow and referencing the dataset's formal identifiers.